# TMDB Keyword ID Crawl

Iterates keyword IDs 1 → ~370,000 via `/3/keyword/{id}` to build a canonical
vocabulary with `tmdb_keyword_id` and `name`.

- Rate limit: 40 req / 10 s
- Checkpoints to `data/tmdb_keywords_checkpoint.json` every 5,000 IDs
- Stops after 2,000 consecutive 404s
- Output: `data/tmdb_keywords_canonical.csv` (`tmdb_keyword_id`, `name`)

Run **cell by cell**; re-running the crawl cell resumes from checkpoint.

In [ ]:
import asyncio
import csv
import json
import os
import time
from pathlib import Path

import aiohttp
from dotenv import load_dotenv

load_dotenv()

TMDB_TOKEN = os.environ['TMDB_TOKEN']
HEADERS = {'Authorization': f'Bearer {TMDB_TOKEN}', 'accept': 'application/json'}
BASE_URL = 'https://api.themoviedb.org/3/keyword/{}'

MAX_ID = 370_000
RATE_LIMIT = 40          # requests per window
RATE_WINDOW = 10.0       # seconds
CONSEC_404_LIMIT = 2_000
CHECKPOINT_EVERY = 5_000 # IDs between checkpoint saves

CHECKPOINT_FILE = Path('data/tmdb_keywords_checkpoint.json')
OUTPUT_FILE = Path('data/tmdb_keywords_canonical.csv')

print('Config OK')

In [ ]:
# Load checkpoint if it exists
if CHECKPOINT_FILE.exists():
    ckpt = json.loads(CHECKPOINT_FILE.read_text())
    start_id = ckpt['next_id']
    results = ckpt['results']  # list of {tmdb_keyword_id, name}
    print(f'Resuming from ID {start_id:,}  ({len(results):,} keywords so far)')
else:
    start_id = 1
    results = []
    print(f'Starting fresh from ID 1')

In [ ]:
async def crawl():
    """Async crawl with a sliding-window rate limiter."""
    # Sliding window: track timestamps of recent requests
    request_times = []
    consecutive_404 = 0
    found = list(results)  # copy checkpoint results
    next_checkpoint = start_id + CHECKPOINT_EVERY

    async def fetch(session, kw_id):
        url = BASE_URL.format(kw_id)
        try:
            async with session.get(url, headers=HEADERS, timeout=aiohttp.ClientTimeout(total=10)) as resp:
                if resp.status == 200:
                    data = await resp.json()
                    return {'tmdb_keyword_id': data['id'], 'name': data['name']}
                elif resp.status == 404:
                    return None
                else:
                    # Transient error — treat as no result but don't count as 404
                    return 'error'
        except Exception:
            return 'error'

    async def rate_limited_fetch(session, kw_id):
        nonlocal request_times
        now = time.monotonic()
        # Drop timestamps older than the window
        request_times = [t for t in request_times if now - t < RATE_WINDOW]
        if len(request_times) >= RATE_LIMIT:
            sleep_for = RATE_WINDOW - (now - request_times[0]) + 0.01
            if sleep_for > 0:
                await asyncio.sleep(sleep_for)
            # Refresh after sleep
            now = time.monotonic()
            request_times = [t for t in request_times if now - t < RATE_WINDOW]
        request_times.append(time.monotonic())
        return await fetch(session, kw_id)

    connector = aiohttp.TCPConnector(limit=RATE_LIMIT)
    async with aiohttp.ClientSession(connector=connector) as session:
        t0 = time.monotonic()
        for kw_id in range(start_id, MAX_ID + 1):
            result = await rate_limited_fetch(session, kw_id)

            if result is None:  # 404
                consecutive_404 += 1
                if consecutive_404 >= CONSEC_404_LIMIT:
                    print(f'\nStopped: {CONSEC_404_LIMIT} consecutive 404s at ID {kw_id:,}')
                    break
            elif result == 'error':
                consecutive_404 = 0  # don't count errors toward 404 streak
            else:
                consecutive_404 = 0
                found.append(result)

            # Progress + checkpoint
            if kw_id % 1_000 == 0:
                elapsed = time.monotonic() - t0
                rate = (kw_id - start_id + 1) / elapsed if elapsed > 0 else 0
                print(f'ID {kw_id:,}  found={len(found):,}  consec404={consecutive_404}  {rate:.0f} req/s', end='\r')

            if kw_id >= next_checkpoint:
                ckpt_data = {'next_id': kw_id + 1, 'results': found}
                CHECKPOINT_FILE.write_text(json.dumps(ckpt_data))
                next_checkpoint = kw_id + CHECKPOINT_EVERY
                print(f'\nCheckpoint saved at ID {kw_id:,}  ({len(found):,} keywords)')

    return found

print('Crawl function defined — run next cell to start')

In [ ]:
# Run crawl (resumable — re-run this cell to continue from checkpoint)
all_keywords = await crawl()
print(f'\nTotal keywords found: {len(all_keywords):,}')

In [ ]:
import pandas as pd

kw_canonical = pd.DataFrame(all_keywords).drop_duplicates('tmdb_keyword_id').sort_values('tmdb_keyword_id').reset_index(drop=True)
OUTPUT_FILE.parent.mkdir(exist_ok=True)
kw_canonical.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(kw_canonical):,} keywords → {OUTPUT_FILE}')
kw_canonical.head(10)

## Join to keyword lexicon

Add `tmdb_keyword_id` to `output/tmdb_keyword_lexicon.csv` by joining on `name`.

In [ ]:
import pandas as pd

lexicon = pd.read_csv('output/tmdb_keyword_lexicon.csv')
canonical = pd.read_csv(OUTPUT_FILE)

# Normalize names for join (lowercase, strip)
canonical['name_norm'] = canonical['name'].str.lower().str.strip()
lexicon['name_norm'] = lexicon['keyword'].str.lower().str.strip()

lexicon = lexicon.merge(
    canonical[['tmdb_keyword_id', 'name_norm']],
    on='name_norm',
    how='left'
).drop(columns='name_norm')

# Move tmdb_keyword_id to second column
cols = ['keyword', 'tmdb_keyword_id'] + [c for c in lexicon.columns if c not in ('keyword', 'tmdb_keyword_id')]
lexicon = lexicon[cols]

matched = lexicon['tmdb_keyword_id'].notna().sum()
print(f'Matched {matched:,} / {len(lexicon):,} keywords ({matched/len(lexicon)*100:.1f}%)')

lexicon.to_csv('output/tmdb_keyword_lexicon.csv', index=False)
print('Updated output/tmdb_keyword_lexicon.csv with tmdb_keyword_id column')
lexicon.head(5)